# Training Experiments

Compare a few controlled MLP experiments using a fixed MNIST training subset and validation split:

- Hidden-layer widths (architecture)
- Learning rate
- Mini-batch size
- ReLU with He initialization vs. sigmoid with Xavier initialization

The model, forward pass, backpropagation, and SGD updates use NumPy. TensorFlow is used only to load MNIST. The official test set is reserved for final evaluation.

These settings are for practical notebook runtime, not a definitive benchmark. Increase the training subset and epochs when running longer experiments.

In [1]:
from pathlib import Path
import time

import numpy as np
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from tensorflow.keras.datasets import mnist

SEED = 42
TRAIN_SUBSET_SIZE = 6_000
VALIDATION_SIZE = 5_000
EPOCHS = 3

rng = np.random.default_rng(SEED)

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find pyproject.toml. Start Jupyter inside the project directory."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy version: {np.__version__}")

Project root: c:\Users\hp\Documents\GitHub\multilayer-perceptron-from-scratch
NumPy version: 2.2.6


## Load data and make fixed, stratified subsets

The validation set and the smaller training subset are sampled only from MNIST's official
training split. Stratification preserves approximate class balance. No test-set images or
labels are loaded here.

In [2]:
(X_all, y_all), _ = mnist.load_data()

all_indices = np.arange(len(y_all))
train_val_indices, val_indices = train_test_split(
    all_indices,
    test_size=VALIDATION_SIZE,
    stratify=y_all,
    random_state=SEED,
)

train_indices, _ = train_test_split(
    train_val_indices,
    train_size=TRAIN_SUBSET_SIZE,
    stratify=y_all[train_val_indices],
    random_state=SEED,
)

X_train, y_train = X_all[train_indices], y_all[train_indices]
X_val, y_val = X_all[val_indices], y_all[val_indices]

print(f"Experiment training subset: {X_train.shape}, labels: {y_train.shape}")
print(f"Validation set:             {X_val.shape}, labels: {y_val.shape}")
print(f"Training class counts:      {np.bincount(y_train, minlength=10)}")
print(f"Validation class counts:    {np.bincount(y_val, minlength=10)}")

Experiment training subset: (6000, 28, 28), labels: (6000,)
Validation set:             (5000, 28, 28), labels: (5000,)
Training class counts:      [592 674 596 613 584 542 592 627 585 595]
Validation class counts:    [494 562 496 511 487 452 493 522 487 496]


## NumPy MLP

The network supports one or more hidden layers, ReLU or sigmoid activations, and He or
Xavier weight initialization. Cross-entropy is calculated from logits for numerical
stability. L2 regularization is optional and disabled by default.

In [3]:
def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=1, keepdims=True)


def cross_entropy_from_logits(
    logits: np.ndarray,
    labels: np.ndarray,
) -> float:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    log_normalizer = np.log(np.sum(np.exp(shifted), axis=1))
    row_indices = np.arange(len(labels))
    return float(np.mean(log_normalizer - shifted[row_indices, labels]))


class NumpyMLP:
    """Fully connected classifier using NumPy only."""

    def __init__(
        self,
        input_dim: int,
        hidden_dims: tuple[int, ...],
        output_dim: int = 10,
        activation: str = "relu",
        initialization: str = "he",
        seed: int = SEED,
    ) -> None:
        if activation not in {"relu", "sigmoid"}:
            raise ValueError("activation must be 'relu' or 'sigmoid'")
        if initialization not in {"he", "xavier"}:
            raise ValueError("initialization must be 'he' or 'xavier'")
        if not hidden_dims or any(width <= 0 for width in hidden_dims):
            raise ValueError("hidden_dims must contain positive layer widths")

        self.activation = activation
        self.layer_dims = (input_dim, *hidden_dims, output_dim)
        self.params: dict[str, np.ndarray] = {}

        init_rng = np.random.default_rng(seed)
        for layer_index, (fan_in, fan_out) in enumerate(
            zip(self.layer_dims[:-1], self.layer_dims[1:]),
            start=1,
        ):
            is_output_layer = layer_index == len(self.layer_dims) - 1
            use_he = initialization == "he" and not is_output_layer
            scale = np.sqrt((2.0 if use_he else 1.0) / fan_in)

            self.params[f"W{layer_index}"] = (
                init_rng.standard_normal((fan_in, fan_out)).astype(np.float32)
                * scale
            )
            self.params[f"b{layer_index}"] = np.zeros(fan_out, dtype=np.float32)

    def _activate(self, values: np.ndarray) -> np.ndarray:
        if self.activation == "relu":
            return np.maximum(values, 0)
        return 1.0 / (1.0 + np.exp(-np.clip(values, -50, 50)))

    def _activation_gradient(self, values: np.ndarray) -> np.ndarray:
        if self.activation == "relu":
            return (values > 0).astype(np.float32)
        activated = self._activate(values)
        return activated * (1.0 - activated)

    def forward(
        self,
        X: np.ndarray,
    ) -> tuple[np.ndarray, tuple[list[np.ndarray], list[np.ndarray]]]:
        activations = [X]
        pre_activations = []
        last_layer = len(self.layer_dims) - 1

        for layer_index in range(1, last_layer + 1):
            values = (
                activations[-1] @ self.params[f"W{layer_index}"]
                + self.params[f"b{layer_index}"]
            )
            pre_activations.append(values)

            if layer_index == last_layer:
                logits = values
            else:
                activations.append(self._activate(values))

        return logits, (activations, pre_activations)

    def loss(
        self,
        X: np.ndarray,
        labels: np.ndarray,
        l2_strength: float = 0.0,
    ) -> float:
        logits, _ = self.forward(X)
        data_loss = cross_entropy_from_logits(logits, labels)
        weight_penalty = sum(
            np.sum(self.params[name] ** 2)
            for name in self.params
            if name.startswith("W")
        )
        return data_loss + 0.5 * l2_strength * float(weight_penalty)

    def backward(
        self,
        labels: np.ndarray,
        cache: tuple[list[np.ndarray], list[np.ndarray]],
        l2_strength: float = 0.0,
    ) -> dict[str, np.ndarray]:
        activations, pre_activations = cache
        layer_count = len(self.layer_dims) - 1
        batch_size = len(labels)

        logits = pre_activations[-1]
        d_values = softmax(logits)
        d_values[np.arange(batch_size), labels] -= 1.0
        d_values /= batch_size

        gradients: dict[str, np.ndarray] = {}

        for layer_index in range(layer_count, 0, -1):
            weights = self.params[f"W{layer_index}"]
            gradients[f"W{layer_index}"] = (
                activations[layer_index - 1].T @ d_values
                + l2_strength * weights
            )
            gradients[f"b{layer_index}"] = np.sum(d_values, axis=0)

            if layer_index > 1:
                d_activations = d_values @ weights.T
                previous_z = pre_activations[layer_index - 2]
                d_values = d_activations * self._activation_gradient(previous_z)

        return gradients

## Batch preprocessing and evaluation

Convert images to flattened float32 vectors normalized to `[0, 1]` as batches are read.
This avoids creating additional full-size floating-point copies of the image arrays.

In [4]:
def make_batch(
    images: np.ndarray,
    labels: np.ndarray,
    indices: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    X_batch = images[indices].reshape(len(indices), -1).astype(np.float32)
    X_batch /= np.float32(255.0)
    y_batch = labels[indices].astype(np.int64)
    return X_batch, y_batch


def evaluate(
    model: NumpyMLP,
    images: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 512,
    l2_strength: float = 0.0,
) -> tuple[float, float]:
    total_loss = 0.0
    total_correct = 0
    total_examples = len(labels)

    for start in range(0, total_examples, batch_size):
        indices = np.arange(start, min(start + batch_size, total_examples))
        X_batch, y_batch = make_batch(images, labels, indices)
        logits, _ = model.forward(X_batch)

        count = len(indices)
        total_loss += cross_entropy_from_logits(logits, y_batch) * count
        total_correct += int(np.sum(np.argmax(logits, axis=1) == y_batch))

    mean_loss = total_loss / total_examples
    if l2_strength:
        weight_penalty = sum(
            np.sum(model.params[name] ** 2)
            for name in model.params
            if name.startswith("W")
        )
        mean_loss += 0.5 * l2_strength * float(weight_penalty)

    return mean_loss, total_correct / total_examples


def train_model(
    model: NumpyMLP,
    X: np.ndarray,
    y: np.ndarray,
    X_validation: np.ndarray,
    y_validation: np.ndarray,
    *,
    epochs: int,
    batch_size: int,
    learning_rate: float,
    l2_strength: float = 0.0,
    seed: int = SEED,
) -> dict[str, list[float]]:
    if batch_size <= 0 or learning_rate <= 0 or epochs <= 0:
        raise ValueError("epochs, batch_size, and learning_rate must be positive")

    shuffle_rng = np.random.default_rng(seed)
    history: dict[str, list[float]] = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    for epoch in range(epochs):
        epoch_start = time.perf_counter()
        order = shuffle_rng.permutation(len(y))

        for start in range(0, len(order), batch_size):
            batch_indices = order[start : start + batch_size]
            X_batch, y_batch = make_batch(X, y, batch_indices)

            _, cache = model.forward(X_batch)
            gradients = model.backward(y_batch, cache, l2_strength)

            for name, parameter in model.params.items():
                parameter -= learning_rate * gradients[name]

        train_loss, train_accuracy = evaluate(
            model, X, y, l2_strength=l2_strength
        )
        val_loss, val_accuracy = evaluate(
            model, X_validation, y_validation, l2_strength=l2_strength
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        elapsed = time.perf_counter() - epoch_start
        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"train acc {train_accuracy:.3f} | "
            f"val acc {val_accuracy:.3f} | "
            f"val loss {val_loss:.3f} | {elapsed:.1f}s"
        )

    return history

## Sanity-check the implementation

Before running the sweeps, verify output shapes, probability normalization, and that
backpropagation produces finite gradients.

In [5]:
tiny_model = NumpyMLP(
    input_dim=784,
    hidden_dims=(16,),
    output_dim=10,
    seed=SEED,
)

X_tiny, y_tiny = make_batch(X_train, y_train, np.arange(8))
logits, cache = tiny_model.forward(X_tiny)
probabilities = softmax(logits)
gradients = tiny_model.backward(y_tiny, cache)

assert logits.shape == (8, 10)
assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
assert all(np.isfinite(gradient).all() for gradient in gradients.values())

print("Forward pass, softmax, and gradient checks passed.")

Forward pass, softmax, and gradient checks passed.


## Run controlled experiments

Each experiment uses the same fixed data subset, validation set, seed, and epoch count.
Change one factor at a time within each comparison group. Results are illustrative and
depend on the selected subset and training budget.

In [6]:
BASE_CONFIG = {
    "hidden_dims": (128,),
    "activation": "relu",
    "initialization": "he",
    "learning_rate": 0.1,
    "batch_size": 128,
    "epochs": EPOCHS,
    "l2_strength": 0.0,
}

results: list[dict[str, object]] = []
histories: dict[str, dict[str, list[float]]] = {}


def run_experiment(
    name: str,
    group: str,
    **overrides: object,
) -> None:
    config = {**BASE_CONFIG, **overrides}
    model = NumpyMLP(
        input_dim=784,
        hidden_dims=config["hidden_dims"],
        output_dim=10,
        activation=config["activation"],
        initialization=config["initialization"],
        seed=SEED,
    )

    start = time.perf_counter()
    history = train_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        learning_rate=config["learning_rate"],
        l2_strength=config["l2_strength"],
        seed=SEED,
    )
    elapsed = time.perf_counter() - start
    histories[name] = history

    parameter_count = sum(parameter.size for parameter in model.params.values())
    results.append(
        {
            "name": name,
            "group": group,
            "hidden_dims": str(config["hidden_dims"]),
            "activation": config["activation"],
            "initialization": config["initialization"],
            "learning_rate": config["learning_rate"],
            "batch_size": config["batch_size"],
            "parameters": parameter_count,
            "val_accuracy": history["val_accuracy"][-1],
            "val_loss": history["val_loss"][-1],
            "seconds": elapsed,
        }
    )

### Architecture comparison

Keep activation, initialization, learning rate, and batch size fixed while varying hidden-layer widths.

In [7]:
for hidden_dims in ((64,), (128,), (128, 64)):
    run_experiment(
        name=f"Architecture {hidden_dims}",
        group="Architecture",
        hidden_dims=hidden_dims,
    )

Epoch 01/3 | train acc 0.824 | val acc 0.808 | val loss 0.827 | 12.7s
Epoch 02/3 | train acc 0.863 | val acc 0.851 | val loss 0.559 | 0.9s
Epoch 03/3 | train acc 0.885 | val acc 0.872 | val loss 0.470 | 0.2s
Epoch 01/3 | train acc 0.827 | val acc 0.818 | val loss 0.791 | 0.3s
Epoch 02/3 | train acc 0.865 | val acc 0.855 | val loss 0.544 | 0.2s
Epoch 03/3 | train acc 0.889 | val acc 0.872 | val loss 0.458 | 0.2s
Epoch 01/3 | train acc 0.816 | val acc 0.801 | val loss 0.702 | 0.2s
Epoch 02/3 | train acc 0.869 | val acc 0.862 | val loss 0.481 | 0.2s
Epoch 03/3 | train acc 0.901 | val acc 0.885 | val loss 0.388 | 0.2s


### Learning-rate comparison

An excessively small rate may learn slowly; an excessively large rate may make optimization unstable.
Compare validation metrics rather than assuming a larger rate is better.

In [8]:
for learning_rate in (0.03, 0.1, 0.3):
    run_experiment(
        name=f"Learning rate {learning_rate}",
        group="Learning rate",
        learning_rate=learning_rate,
    )

Epoch 01/3 | train acc 0.696 | val acc 0.683 | val loss 1.582 | 0.3s
Epoch 02/3 | train acc 0.779 | val acc 0.770 | val loss 1.094 | 0.3s
Epoch 03/3 | train acc 0.820 | val acc 0.811 | val loss 0.841 | 0.2s
Epoch 01/3 | train acc 0.827 | val acc 0.818 | val loss 0.791 | 0.2s
Epoch 02/3 | train acc 0.865 | val acc 0.855 | val loss 0.544 | 0.2s
Epoch 03/3 | train acc 0.889 | val acc 0.872 | val loss 0.458 | 0.2s
Epoch 01/3 | train acc 0.875 | val acc 0.862 | val loss 0.481 | 0.2s
Epoch 02/3 | train acc 0.894 | val acc 0.880 | val loss 0.400 | 0.2s
Epoch 03/3 | train acc 0.920 | val acc 0.901 | val loss 0.337 | 0.3s


### Batch-size comparison

Batch size changes the number of parameter updates per epoch and the amount of data used
to estimate each gradient.

In [9]:
for batch_size in (32, 128, 512):
    run_experiment(
        name=f"Batch size {batch_size}",
        group="Batch size",
        batch_size=batch_size,
    )

Epoch 01/3 | train acc 0.886 | val acc 0.875 | val loss 0.431 | 0.6s
Epoch 02/3 | train acc 0.887 | val acc 0.875 | val loss 0.413 | 0.4s
Epoch 03/3 | train acc 0.918 | val acc 0.891 | val loss 0.345 | 0.5s
Epoch 01/3 | train acc 0.827 | val acc 0.818 | val loss 0.791 | 0.2s
Epoch 02/3 | train acc 0.865 | val acc 0.855 | val loss 0.544 | 0.2s
Epoch 03/3 | train acc 0.889 | val acc 0.872 | val loss 0.458 | 0.2s
Epoch 01/3 | train acc 0.666 | val acc 0.655 | val loss 1.683 | 0.2s
Epoch 02/3 | train acc 0.750 | val acc 0.739 | val loss 1.214 | 0.2s
Epoch 03/3 | train acc 0.806 | val acc 0.799 | val loss 0.936 | 0.2s


### Activation and initialization comparison

Pair ReLU with He initialization and sigmoid with Xavier initialization. This compares
two reasonable combinations; it is not a full factorial study of every activation/init pair.

In [10]:
run_experiment(
    name="ReLU + He",
    group="Activation / initialization",
    activation="relu",
    initialization="he",
)

run_experiment(
    name="Sigmoid + Xavier",
    group="Activation / initialization",
    activation="sigmoid",
    initialization="xavier",
)

Epoch 01/3 | train acc 0.827 | val acc 0.818 | val loss 0.791 | 0.3s
Epoch 02/3 | train acc 0.865 | val acc 0.855 | val loss 0.544 | 0.3s
Epoch 03/3 | train acc 0.889 | val acc 0.872 | val loss 0.458 | 0.3s
Epoch 01/3 | train acc 0.650 | val acc 0.636 | val loss 1.959 | 0.3s
Epoch 02/3 | train acc 0.652 | val acc 0.647 | val loss 1.592 | 0.3s
Epoch 03/3 | train acc 0.765 | val acc 0.759 | val loss 1.268 | 0.3s


## Compare validation results

Validation accuracy and loss are useful for choosing settings. They are not substitutes
for a final evaluation on the reserved official test set.

In [11]:
sorted_results = sorted(
    results,
    key=lambda row: (str(row["group"]), -float(row["val_accuracy"])),
)

header = (
    f"{'Group':<28} {'Experiment':<32} {'Val acc':>9} "
    f"{'Val loss':>10} {'Seconds':>9}"
)
print(header)
print("-" * len(header))

for row in sorted_results:
    print(
        f"{str(row['group']):<28} "
        f"{str(row['name']):<32} "
        f"{float(row['val_accuracy']):>8.3%} "
        f"{float(row['val_loss']):>10.4f} "
        f"{float(row['seconds']):>8.1f}"
    )

Group                        Experiment                         Val acc   Val loss   Seconds
--------------------------------------------------------------------------------------------
Activation / initialization  ReLU + He                         87.200%     0.4582      0.8
Activation / initialization  Sigmoid + Xavier                  75.860%     1.2679      0.8
Architecture                 Architecture (128, 64)            88.520%     0.3876      0.7
Architecture                 Architecture (64,)                87.200%     0.4698     13.8
Architecture                 Architecture (128,)               87.200%     0.4582      0.7
Batch size                   Batch size 32                     89.100%     0.3446      1.5
Batch size                   Batch size 128                    87.200%     0.4582      0.7
Batch size                   Batch size 512                    79.900%     0.9361      0.6
Learning rate                Learning rate 0.3                 90.140%     0.3366     

In [12]:
fig = go.Figure()

for group in dict.fromkeys(str(row["group"]) for row in results):
    group_rows = [row for row in results if row["group"] == group]
    fig.add_trace(
        go.Bar(
            name=group,
            x=[str(row["name"]) for row in group_rows],
            y=[float(row["val_accuracy"]) for row in group_rows],
        )
    )

fig.update_layout(
    title="Validation accuracy by experiment",
    xaxis_title="Experiment",
    yaxis_title="Validation accuracy",
    yaxis=dict(tickformat=".0%"),
    barmode="group",
    xaxis_tickangle=-30,
)
fig.show()

In [13]:
fig = go.Figure()

for group in dict.fromkeys(str(row["group"]) for row in results):
    group_rows = [row for row in results if row["group"] == group]
    fig.add_trace(
        go.Bar(
            name=group,
            x=[str(row["name"]) for row in group_rows],
            y=[float(row["val_loss"]) for row in group_rows],
        )
    )

fig.update_layout(
    title="Validation loss by experiment",
    xaxis_title="Experiment",
    yaxis_title="Cross-entropy loss",
    barmode="group",
    xaxis_tickangle=-30,
)
fig.show()

## Interpretation and next steps

Use the results to discuss:

- Which hidden-layer widths gave the best validation results, and how did parameter count change?
- Did learning rate affect convergence or stability?
- How did batch size affect training time and validation performance?
- How did ReLU + He compare with sigmoid + Xavier?

These short runs on a subset are exploratory. Confirm promising settings with more training data
and epochs, then evaluate the selected model once on the official test set. Record the seed,
data split, hyperparameters, and environment when reporting results.

Next: inspect learned weights and model predictions in `03_Interpretability.ipynb`.